Sprint 1 - Importando os Dados.

In [22]:

#Importando os Dados:
import pandas as pd
import numpy as np
import csv
from IPython.display import display

caminho = "dataset/Base Varejo.csv"

with open(caminho, "r", encoding="utf-8") as arquivo:
    leitor = csv.DictReader(arquivo, delimiter=";") # sem o delimiter a base aparece como uma única coluna. Isso indica que o CSV utiliza ";" como separador.
    dados = list(leitor)

varejo = pd.DataFrame(dados)

print("Informações da Base de Dados")
print("Quantidade de registros:", len(varejo))
print("Quantidade de colunas:", len(varejo.columns))

print("Colunas:")
print(varejo.columns.tolist())

print("Tipos de dados:")
print(varejo.dtypes)

print("Primeiras linhas:")
display(varejo.head())

Informações da Base de Dados
Quantidade de registros: 830000
Quantidade de colunas: 11
Colunas:
['DATA', 'CO_ID', 'CL_ID', 'CL_GENERO', 'CL_EC', 'CL_FHL', 'CL_SEG', 'PR_ID', 'PR_CAT', 'PR_NOME', '']
Tipos de dados:
DATA         str
CO_ID        str
CL_ID        str
CL_GENERO    str
CL_EC        str
CL_FHL       str
CL_SEG       str
PR_ID        str
PR_CAT       str
PR_NOME      str
             str
dtype: object
Primeiras linhas:


,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,
0,01/02/2019,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA,
1,01/02/2019,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS,
2,01/02/2019,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO,
3,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI,
4,01/02/2019,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO,


quando usamos o csv.DictReader, o shape ficou com 11 colunas porque o arquivo possui colunas extras sem nome no final do cabeçalho. Como o DictReader transforma cada linha em um dicionário, colunas com o mesmo nome vazio não são mantidas separadamente, ele junta tudo numa coluna " "  como podemos ver no tipo de dados após o PR_NOME. 

In [35]:
#para verificar o cabeçalho real: 
import csv

arquivo = "dataset/Base Varejo.csv"

with open(arquivo, mode="r", encoding="utf-8-sig", newline="") as csvfile:
    leitor_csv = csv.reader(csvfile, delimiter=";")
    cabecalho = next(leitor_csv)

print("Quantidade de colunas no cabeçalho bruto:")
print(len(cabecalho))

print("Cabeçalho bruto:")
print(cabecalho)

Quantidade de colunas no cabeçalho bruto:
14
Cabeçalho bruto:
['DATA', 'CO_ID', 'CL_ID', 'CL_GENERO', 'CL_EC', 'CL_FHL', 'CL_SEG', 'PR_ID', 'PR_CAT', 'PR_NOME', '', '', '', '']


Como citado a cima, aqui aparecem as 4 colunas vazias, que se juntam no DictRead de forma automática transformando em apenas 1 coluna " " 

Sprint 2  - Transformação dos Dados.

In [36]:

# Limpando e padronizando as Strings
# aqui estamos criando uma lista chamada colunas_texto, que contem todas essas colunas dentro desta lista.
colunas_texto = [
    "CL_GENERO",
    "CL_EC",
    "CL_FHL",
    "PR_CAT",
    "PR_NOME"
]

for coluna in colunas_texto:
    varejo[coluna] = (
        varejo[coluna]
        .astype(str)                             # garante string mesmo se vier tipo misto
        .str.strip()                             # remove os espaços nas pontas
        .str.replace(r"\s+", " ", regex=True)    # remove os espaços internos duplicados ex: " João   Silva" -> "João Silva"
        .replace("NAN", pd.NA)                   # astype(str) transforma NaN em "nan" -> corrige aqui
    )

#Convertendo colunas numéricas inteiras
#aqui é o mesmo processo de criar a lista com as colunas desejadas dentro dela. 
colunas_num_inteiras = [
    "CO_ID",
    "CL_ID",
    "PR_ID"
]
#a função for percorre cada item de cada lista e retorna com a conversão de cada item de cada coluna, um de cada vez. 
for coluna in colunas_num_inteiras:
    varejo[coluna] = pd.to_numeric(
        varejo[coluna],
        errors="coerce"
    ).astype("Int64")

    n_invalidos = varejo[coluna].isna().sum()
    if n_invalidos > 0:
        print(f"[Aviso] {coluna}: {n_invalidos} valores não convertidos (viraram NA)")

#Convertendo a data para o padrão:

varejo["DATA"] = pd.to_datetime(
    varejo["DATA"],
    errors="coerce",
    dayfirst=True                      # transforma para o padrão brasileiro com os dias primeiro. ex dd/mm/aaaa
)
#aqui é um mecanismo para o sistema informar caso tenha uma data equivocada que não conseguiu ser convertida.
n_datas_invalidas = varejo["DATA"].isna().sum()
if n_datas_invalidas > 0:
    print(f"[Aviso] DATA: {n_datas_invalidas} valores não convertidos (viraram NaT)")

#Resultado das colunas depois das transformações
print("Resultados das colunas depois das transformações:")
print(varejo.dtypes)

Resultados das colunas depois das transformações:
DATA         datetime64[us]
CO_ID                 Int64
CL_ID                 Int64
CL_GENERO               str
CL_EC                   str
CL_FHL                  str
CL_SEG              float64
PR_ID                 Int64
PR_CAT                  str
PR_NOME                 str
                        str
dtype: object


Perceba que a coluna 11 que está vazia permanece. vamos remover ela no próximo Sprint.

Sprint 3 - Removendo Nulos e duplicatas.

In [42]:
#Vamos verificar os valores nulos com o isnull e depois somar todos os nulos com a função .sum()

print("Valores nulos em cada coluna:")
nulos = varejo.isnull().sum()    
print(nulos[nulos > 0])


#Conferindo as categorias vazias.
print("Categorias vazias:")

categorias_vazias = (
    varejo["PR_CAT"].isna() |
    (varejo["PR_CAT"].str.strip() == "")
).sum()

print("Categorias vazias:", categorias_vazias)

#Tratando as categorias vazias.

varejo["PR_CAT"] = varejo["PR_CAT"].fillna("SEM CATEGORIA")
varejo["PR_CAT"] = varejo["PR_CAT"].replace("", "SEM CATEGORIA")


#Tratando as categorias completamente vazias, onde a coluna inteira está vazia.
colunas_vazias = varejo.columns[varejo.isna().all()]

print("A Coluna inteira está vazia:")
print(colunas_vazias.tolist())


#Remove as colunas que possuem todos os valores nulos
varejo = varejo.dropna(axis=1, how="all")


#vamos conferir as duplicatas agora. 

duplicatas = varejo.duplicated().sum()
print("Duplicatas encontradas:", duplicatas)

#Agora vamos remover as duplicatas.
varejo = varejo.drop_duplicates()

print("Duplicatas após a limpeza:",
      varejo.duplicated().sum())


#Conferindo se existe mais algum nulo após a limpeza.

print("Nulos após a limpeza:")
nulos_final = varejo.isnull().sum()

print(nulos_final[nulos_final > 0])



Valores nulos em cada coluna:
Series([], dtype: int64)
Categorias vazias:
Categorias vazias: 0
A Coluna inteira está vazia:
[]
Duplicatas encontradas: 0
Duplicatas após a limpeza: 0
Nulos após a limpeza:
Series([], dtype: int64)
